In [2]:
import time
import pandas as pd
from nba_api.stats.endpoints import synergyplaytypes

In [ ]:
play_types = ['Isolation', 'Transition', 'PRBallHandler', 'PRRollman', 'Postup',
              'Spotup', 'Handoff', 'Cut', 'OffScreen', 'OffRebound', 'Misc']
frames = []
for pt in play_types:
    frames.append(synergyplaytypes.SynergyPlayTypes(
        season='2025-26',
        season_type_all_star='Regular Season',
        player_or_team_abbreviation='P',
        type_grouping_nullable='offensive',
        play_type_nullable=pt,
        per_mode_simple='Totals',
    ).get_data_frames()[0])
    time.sleep(1)

synergy = pd.concat(frames)
synergy.to_csv('../../data/raw/nba-api/synergy_offense_2025-26.csv', index=False)

freq = synergy.pivot_table(index=['PLAYER_ID', 'PLAYER_NAME'], columns='PLAY_TYPE',
                           values='POSS_PCT', fill_value=0)

# POSS_PCT in this case tells us of all the possesions a player used (ending with a shot, a turnover,
# or a foul drawn), all the percentages they ended it with.

In [ ]:
def_types = ['Isolation', 'PRBallHandler', 'PRRollman', 'Postup', 'Spotup', 'Handoff', 'OffScreen']
frames = []
for pt in def_types:
    frames.append(synergyplaytypes.SynergyPlayTypes(
        season='2025-26',
        season_type_all_star='Regular Season',
        player_or_team_abbreviation='P',
        type_grouping_nullable='defensive',
        play_type_nullable=pt,
        per_mode_simple='Totals',
    ).get_data_frames()[0])
    time.sleep(1)

synergy_def = pd.concat(frames)
synergy_def.to_csv('../../data/raw/nba-api/synergy_defense_2025-26.csv', index=False)

def_freq = synergy_def.pivot_table(index=['PLAYER_ID', 'PLAYER_NAME'], columns='PLAY_TYPE',
                                   values='POSS_PCT', fill_value=0).add_prefix('def_')


# POSS_PCT is the share of the plays this player was credited with defending
# that specific type. It'll be useful in showing a defensive role and maybe even defensive exploitation
# where we can see which teams target some specific players in actions (Maybe Jokic in pnr for example)

# PPP is points allowed per possession

In [ ]:
from nba_api.stats.endpoints import shotchartdetail

# The endpoint silently caps a response at ~100k rows (only half a season),
# so pull one month at a time and stitch the months together
months = pd.date_range('2025-10-01', '2026-05-01', freq='MS')
frames = []
for start, end in zip(months[:-1], months[1:] - pd.Timedelta(days=1)):
    frames.append(shotchartdetail.ShotChartDetail(
        team_id=0, player_id=0,
        season_nullable='2025-26',
        season_type_all_star='Regular Season',
        context_measure_simple='FGA',
        date_from_nullable=start.strftime('%m/%d/%Y'),
        date_to_nullable=end.strftime('%m/%d/%Y'),
    ).get_data_frames()[0])
    time.sleep(1)

shots = pd.concat(frames, ignore_index=True)
shots.to_csv('../../data/raw/nba-api/shots_2025-26.csv', index=False)
print(len(shots), 'shots,', shots['GAME_ID'].nunique(), 'games')  # full season = 1230 games


# Shot type distribution per player (style features)
action_freq = pd.crosstab([shots['PLAYER_ID'], shots['PLAYER_NAME']], shots['ACTION_TYPE'], normalize='index')
zone_freq = pd.crosstab([shots['PLAYER_ID'], shots['PLAYER_NAME']], shots['SHOT_ZONE_AREA'], normalize='index')

In [8]:
zone_freq

,SHOT_ZONE_AREA,Back Court(BC),Center(C),Left Side Center(LC),Left Side(L),Right Side Center(RC),Right Side(R)
PLAYER_ID,PLAYER_NAME,,,,,,
2544,LeBron James,0.0,0.590909,0.146104,0.113636,0.087662,0.061688
101108,Chris Paul,0.0,0.410714,0.196429,0.071429,0.178571,0.142857
200768,Kyle Lowry,0.0,0.285714,0.285714,0.142857,0.142857,0.142857
201142,Kevin Durant,0.0,0.489933,0.145973,0.119128,0.129195,0.115772
201143,Al Horford,0.0,0.375000,0.139706,0.095588,0.220588,0.169118
...,...,...,...,...,...,...,...
1642962,Drake Powell,0.0,0.504202,0.084034,0.134454,0.168067,0.109244
1642964,Brooks Barnhizer,0.0,0.740741,0.037037,0.074074,0.037037,0.111111
1643007,Taelon Peter,0.0,0.147059,0.235294,0.176471,0.323529,0.117647


In [9]:
action_freq

,ACTION_TYPE,Alley Oop Dunk Shot,Alley Oop Layup shot,Cutting Dunk Shot,Cutting Finger Roll Layup Shot,Cutting Layup Shot,Driving Bank Hook Shot,Driving Dunk Shot,Driving Finger Roll Layup Shot,Driving Floating Bank Jump Shot,Driving Floating Jump Shot,...,Step Back Bank Jump Shot,Step Back Jump shot,Tip Dunk Shot,Tip Layup Shot,Turnaround Bank Hook Shot,Turnaround Bank shot,Turnaround Fadeaway Bank Jump Shot,Turnaround Fadeaway shot,Turnaround Hook Shot,Turnaround Jump Shot
PLAYER_ID,PLAYER_NAME,,,,,,,,,,,,,,,,,,,,,
2544,LeBron James,0.012987,0.003247,0.012987,0.006494,0.009740,0.0,0.012987,0.012987,0.012987,0.029221,...,0.0,0.042208,0.000000,0.006494,0.0,0.0,0.003247,0.071429,0.003247,0.019481
101108,Chris Paul,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.017857,0.000000,0.017857,...,0.0,0.017857,0.000000,0.000000,0.0,0.0,0.000000,0.017857,0.000000,0.000000
200768,Kyle Lowry,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000
201142,Kevin Durant,0.000000,0.000000,0.006711,0.001678,0.006711,0.0,0.018456,0.005034,0.015101,0.043624,...,0.0,0.021812,0.000000,0.001678,0.0,0.0,0.000000,0.062081,0.000000,0.023490
201143,Al Horford,0.000000,0.007353,0.000000,0.000000,0.014706,0.0,0.000000,0.000000,0.014706,0.007353,...,0.0,0.014706,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.022059,0.007353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1642962,Drake Powell,0.000000,0.000000,0.008403,0.000000,0.008403,0.0,0.008403,0.000000,0.000000,0.084034,...,0.0,0.067227,0.000000,0.000000,0.0,0.0,0.000000,0.016807,0.000000,0.025210
1642964,Brooks Barnhizer,0.000000,0.000000,0.074074,0.000000,0.037037,0.0,0.000000,0.074074,0.000000,0.037037,...,0.0,0.000000,0.037037,0.037037,0.0,0.0,0.000000,0.000000,0.000000,0.111111
1643007,Taelon Peter,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,0.0,0.058824,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000
